# Phase 7 — 로컬 Qwen LLM 전환

Claude API → Ollama + Qwen2.5 로컬 LLM으로 전환하고, 실제 RAG 품질을 비교합니다.

## 전제 조건
```bash
# 1. Ollama 설치 및 모델 다운로드 (CPU 환경: 3b 권장)
chmod +x scripts/setup_ollama.sh && ./scripts/setup_ollama.sh

# Windows의 경우:
# https://ollama.com/download 에서 설치 후
# ollama pull qwen2.5:3b

# 2. Ollama 서버 실행
# ollama serve
```

## 환경별 권장 모델
| 환경 | 권장 모델 | 디스크 | 응답속도 |
|------|-----------|--------|----------|
| CPU only | qwen2.5:3b | ~2GB | 느림 (30~120초) |
| GPU 4-8GB | qwen2.5:7b | ~5GB | 보통 (5~15초) |
| GPU 16GB+ | qwen2.5:14b | ~9GB | 빠름 (2~5초) |

In [ ]:
# ─────────────────────────────────────────────
# 셀 0: 환경 설정 (CPU/GPU 공통)
# ─────────────────────────────────────────────
import sys, os, time, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from dotenv import load_dotenv
load_dotenv('../.env')

print('✅ 환경 설정 완료')
print(f'   LLM_PROVIDER: {os.getenv("LLM_PROVIDER", "anthropic (기본)")}' )

In [ ]:
# ─────────────────────────────────────────────
# 셀 1: Ollama 서버 연결 확인 (CPU/GPU 공통)
# ─────────────────────────────────────────────
import httpx

OLLAMA_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434')
OLLAMA_MODEL = os.getenv('OLLAMA_MODEL', 'qwen2.5:3b')

try:
    resp = httpx.get(f'{OLLAMA_URL}/api/tags', timeout=5)
    models = [m['name'] for m in resp.json().get('models', [])]
    print(f'✅ Ollama 연결 성공: {OLLAMA_URL}')
    print(f'   설치된 모델: {models}')
    OLLAMA_AVAILABLE = OLLAMA_MODEL in models
    if not OLLAMA_AVAILABLE:
        print(f'⚠️  {OLLAMA_MODEL} 모델이 없습니다. 아래 명령으로 설치하세요:')
        print(f'   ollama pull {OLLAMA_MODEL}')
except Exception as e:
    print(f'❌ Ollama 연결 실패: {e}')
    print('   ollama serve 명령어로 서버를 먼저 실행해주세요.')
    OLLAMA_AVAILABLE = False

In [ ]:
# ─────────────────────────────────────────────
# 셀 2: 벡터 스토어 및 Retriever 초기화 (CPU/GPU 공통)
# ─────────────────────────────────────────────
from src.embeddings import EmbeddingManager
from src.vectorstore import PostgresVectorStore

embedding_provider = 'openai' if os.getenv('OPENAI_API_KEY') else 'huggingface'
embeddings = EmbeddingManager(provider=embedding_provider).embeddings
print(f'임베딩 제공자: {embedding_provider}')

vs = PostgresVectorStore(embeddings, collection_name='notion_docs')
retriever = vs.as_retriever(search_kwargs={'k': 4}, score_threshold=0.3)

stats = vs.get_collection_stats()
print(f'벡터 스토어 문서 수: {stats["count"]}개')

In [ ]:
# ─────────────────────────────────────────────
# 셀 3: Claude vs Qwen RAG 체인 초기화 (CPU/GPU 공통)
# ─────────────────────────────────────────────
from src.llm import LLMAdapter
from src.chains import RAGChain

# Claude (기존 API 기반)
claude_llm = LLMAdapter(provider='anthropic', temperature=0).llm
claude_chain = RAGChain(claude_llm, retriever)
print('✅ Claude 체인 초기화 완료')

# Qwen via Ollama (로컬)
if OLLAMA_AVAILABLE:
    qwen_llm = LLMAdapter(provider='ollama', model_name=OLLAMA_MODEL, temperature=0).llm
    qwen_chain = RAGChain(qwen_llm, retriever)
    print(f'✅ Qwen 체인 초기화 완료 (모델: {OLLAMA_MODEL})')
else:
    print('⚠️  Ollama 미연결 — Claude 단독 실행')

In [ ]:
# ─────────────────────────────────────────────
# 셀 4: Claude vs Qwen 질의 비교 (CPU/GPU 공통)
# ─────────────────────────────────────────────
# 아래 질문을 실제 Notion 문서 내용 기반 질문으로 수정하세요
test_questions = [
    '이 지식베이스의 주요 내용을 한 문단으로 요약해주세요.',
    '가장 최근에 다룬 기술적 주제는 무엇인가요?',
    '2050년의 우주여행 방법을 알려주세요.',  # out-of-scope 테스트
]

comparison_results = []

for q in test_questions:
    print(f'\n질문: {q}')
    print('─' * 60)

    row = {'question': q}

    # Claude
    t0 = time.time()
    claude_ans = claude_chain.invoke(q)
    claude_time = round(time.time() - t0, 2)
    print(f'[Claude] ({claude_time}초)\n{claude_ans[:200]}{"..." if len(claude_ans) > 200 else ""}')
    row['claude'] = {'answer': claude_ans, 'latency': claude_time}

    # Qwen
    if OLLAMA_AVAILABLE:
        t0 = time.time()
        qwen_ans = qwen_chain.invoke(q)
        qwen_time = round(time.time() - t0, 2)
        print(f'\n[Qwen {OLLAMA_MODEL}] ({qwen_time}초)\n{qwen_ans[:200]}{"..." if len(qwen_ans) > 200 else ""}')
        row['qwen'] = {'answer': qwen_ans, 'latency': qwen_time}

    comparison_results.append(row)

print('\n✅ 비교 완료')

In [ ]:
# ─────────────────────────────────────────────
# 셀 5: 결과 요약 및 저장 (CPU/GPU 공통)
# ─────────────────────────────────────────────
print('=' * 60)
print('비교 결과 요약')
print('=' * 60)

claude_times = [r['claude']['latency'] for r in comparison_results if 'claude' in r]
qwen_times   = [r['qwen']['latency']   for r in comparison_results if 'qwen'   in r]

print(f'{"항목":<22} {"Claude":>16} {"Qwen " + OLLAMA_MODEL:>20}')
print('─' * 60)
print(f'{"평균 응답 시간":<22} {sum(claude_times)/len(claude_times):>15.2f}초 {(sum(qwen_times)/len(qwen_times) if qwen_times else 0):>19.2f}초')
print(f'{"API 비용":<22} {"유료":>16} {"무료":>20}')
print(f'{"인터넷 필요":<22} {"O":>16} {"X (완전 오프라인)":>20}')
print('=' * 60)

# 결과 저장
out = Path('../data/llm_comparison.json')
out.parent.mkdir(exist_ok=True)
with open(out, 'w', encoding='utf-8') as f:
    json.dump(comparison_results, f, ensure_ascii=False, indent=2)
print(f'\n결과 저장: {out}')

In [ ]:
# ─────────────────────────────────────────────
# 셀 6: 전환 방법 안내 (CPU/GPU 공통)
# ─────────────────────────────────────────────
print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Ollama로 완전 전환하는 방법
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

.env 파일 수정:
  LLM_PROVIDER=ollama
  OLLAMA_BASE_URL=http://localhost:11434
  OLLAMA_MODEL=qwen2.5:3b

  # Ollama 실패 시 Claude로 자동 전환 (선택)
  FALLBACK_LLM_PROVIDER=anthropic

서비스 실행:
  uvicorn api.server:app --host 0.0.0.0 --port 8000
  python app.py

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 다음 단계: 04_lora_tuning.ipynb
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
GPU 환경에서 Qwen2.5를 Notion 문서 도메인으로 LoRA 파인튜닝합니다.
""")